In [22]:
# Required packages
import numpy as np
import pandas as pd
import os

In [23]:
# Define path and environment
input_path = "/Users/minjuhee/Desktop/HPLC/7_Jangcheon/3_Phenology"
input_file = "dino_functrait.xlsx"
output_file = "dino_functrait_calculated.xlsx"
sheet_name = "FuncTrait32"

In [27]:
# Useful function
# assignment of size-classes
def assign_sizefrac(esd):
    """Assign phytoplankton size class from ESD (µm)."""
    if pd.isna(esd):
        return np.nan
    elif esd < 2:
        return "pico"
    elif esd < 20:
        return "nano"
    elif esd < 200:
        return "micro"
    elif esd < 2000:
        return "meso"
    else:
        return "mega"

# calculate biovolume for one row
def calc_biovolume(row):
    """
    Calculate biovolume based on Length, Width, Geometric shape, HD, and Fraction.
    Includes special handling for 'Irregular' shapes (Ceratium / Tripos).
    """
    length = row["Length"]
    width = row["Width"]
    shape = row["Geometric shape"]
    hd = row["HD"]
    frac = row["Fraction"]
    name = str(row.get("Scientific name", "")).lower()

    # Default fraction = 1 if missing or 0
    if pd.isna(frac) or frac == 0:
        frac = 1.0

    d = width
    h1 = length

    # Mean diameter for sphere-type
    d_mean = (width + length) / 2

    # Flattened ellipsoid parameters
    d1 = width
    d2 = hd * d1 if pd.notna(hd) else d1 / 2

    # -----------------------------
    # Standard geometric shapes
    # -----------------------------
    if shape == "Sphere":
        volume = (np.pi / 6) * (d_mean ** 3)

    elif shape == "Half Sphere":
        volume = (np.pi / 12) * (d_mean ** 3)

    elif shape == "Cylinder":
        volume = (np.pi / 4) * (d ** 2) * h1

    elif shape == "Rotational Ellipsoid":
        volume = (np.pi / 6) * (d ** 2) * h1

    elif shape == "Flattened Ellipsoid":
        volume = (np.pi / 6) * d1 * d2 * h1

    elif shape == "Cone":
        volume = (np.pi / 12) * (d ** 2) * h1

    elif shape == "Half Cone":
        volume = (np.pi / 24) * (d ** 2) * h1

    elif shape == "Double Cone":
        volume = (np.pi / 12) * (d ** 2) * h1

    elif shape == "Cone + Half Sphere":
        volume = (np.pi / 12) * (d ** 2) * h1

    elif shape == "Half Cone + Cut Flattened Ellipsoid":
        h2 = h1 / 2
        volume = (np.pi / 24) * (d1 ** 2) * h2 + (np.pi / 6) * d1 * d2 * h1

    elif shape == "Irregular":

        if "ceratium furca" in name or "ceratium kofoidii" in name:
            volume = 2.3038 * (d ** 2.532)

        elif "ceratium fusus" in name:
            volume = 35.198 * (d ** 1.9156)

        elif "ceratium tripos" in name:
            volume = 0.32359 * (d ** 2.9953)

        else:
            return np.nan

    else:
        return np.nan

    # Apply fraction at the end
    return volume * frac

In [28]:
# Prepare input data
df = pd.read_excel(os.path.join(input_path, input_file), sheet_name=sheet_name)

for col in ["Length", "Width", "HD", "Fraction"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Geometric shape"] = df["Geometric shape"].astype(str).str.strip()

In [31]:
# Run
df["Biovolume"] = df.apply(calc_biovolume, axis=1)

df["ESD"] = np.where(
    df["Biovolume"] > 0,
    (6 * df["Biovolume"] / np.pi) ** (1 / 3),
    np.nan
)

df["SizeFrac"] = df["ESD"].apply(assign_sizefrac)

# Check a few important columns
df[["Scientific name", "Length", "Width", "Geometric shape", "HD", "Fraction", "Biovolume", "ESD", "SizeFrac"]].head(32)

,Scientific name,Length,Width,Geometric shape,HD,Fraction,Biovolume,ESD,SizeFrac
0,Alexandrium sp.,35.10,34.85,Rotational Ellipsoid,NaN,0.00,2.232088e+04,34.933135,micro
1,Akashiwo sanguinea,70.00,50.00,Flattened Ellipsoid,NaN,NaN,4.581489e+04,44.395200,micro
2,Amylax sp.,45.50,28.75,Cone + Half Sphere,NaN,0.75,7.384430e+03,24.160511,micro
3,Ceratium furca (Tripos furca),NaN,42.95,Irregular,NaN,0.00,3.141286e+04,39.147404,micro
4,Ceratium fusus (Tripos fusus),NaN,20.95,Irregular,NaN,0.00,1.195028e+04,28.365678,micro
5,Ceratium kofoidii (Tripos kofoidii),78.50,18.50,Irregular,NaN,0.00,3.723262e+03,19.229810,nano
6,Ceratium tripos (Tripos muelleri),71.50,56.50,Irregular,NaN,0.00,5.726718e+04,47.822880,micro
7,Cochlodinium sp. (Margalefidinium polykrikoides),32.75,26.15,Rotational Ellipsoid,NaN,NaN,1.172609e+04,28.187176,micro
8,Dinophysis acuminata,48.00,32.00,Flattened Ellipsoid,0.50,0.00,1.286796e+04,29.073929,micro
9,Dinophysis caudata,86.00,41.20,Flattened Ellipsoid,0.45,0.00,3.439569e+04,40.349219,micro


In [32]:
with pd.ExcelWriter(os.path.join(input_path, output_file), engine="openpyxl", mode="w") as writer:
    # Write updated FuncTrait32 sheet
    df.to_excel(writer, sheet_name=sheet_name, index=False)

    # Also copy the other sheets from the original workbook
    xls = pd.ExcelFile(input_file)
    for s in xls.sheet_names:
        if s != sheet_name:
            temp_df = pd.read_excel(input_file, sheet_name=s)
            temp_df.to_excel(writer, sheet_name=s, index=False)

print(f"Done. Updated file saved as: {output_file}")

Done. Updated file saved as: dino_functrait_calculated.xlsx
